## Projeto Final - Plano de Trabalho
### Caso de Estudo: Industria Steelproof

A planta siderúrgica Steelproof busca reduzir custos de produção por meio de otimizaçao do consumo de energia durante o processamento do aço.  O objetivo deste projeto é desenvolver um modelo capaz de prever a temperatura do metal, apoiando decisões mais eficientes e sustentáveis no processo industrial.

Este notebook tem como objetivo apresentar uma análise exploratória inicial dos dados, levantar dúvidas relevantes sobre o problema e propor um plano geralde resolução da tarefa.

In [1]:
# previa dos dados
import pandas as pd

arc = pd.read_csv('data_arc_en.csv')
gas = pd.read_csv('data_gas_en.csv')
temp = pd.read_csv('data_temp_en.csv')

arc.head(), arc.info()

<class 'pandas.core.frame.DataFrame'>
,RangeIndex: 14876 entries, 0 to 14875
,Data columns (total 5 columns):
, #   Column             Non-Null Count  Dtype  
,---  ------             --------------  -----  
, 0   key                14876 non-null  int64  
, 1   Arc heating start  14876 non-null  object 
, 2   Arc heating end    14876 non-null  object 
, 3   Active power       14876 non-null  float64
, 4   Reactive power     14876 non-null  float64
,dtypes: float64(2), int64(1), object(2)
,memory usage: 581.2+ KB


(   key    Arc heating start      Arc heating end  Active power  Reactive power
 0    1  2019-05-03 11:02:14  2019-05-03 11:06:02      0.976059        0.687084
 1    1  2019-05-03 11:07:28  2019-05-03 11:10:33      0.805607        0.520285
 2    1  2019-05-03 11:11:44  2019-05-03 11:14:36      0.744363        0.498805
 3    1  2019-05-03 11:18:14  2019-05-03 11:24:19      1.659363        1.062669
 4    1  2019-05-03 11:26:09  2019-05-03 11:28:37      0.692755        0.414397,
 None)

In [2]:
# dados ausentes
print("Keys únicas em arc:", arc['key'].nunique())
print("Keys únicas em gas:", gas['key'].nunique())
print("Keys únicas em temp:", temp['key'].nunique())

Keys únicas em arc: 3214
,Keys únicas em gas: 3239
,Keys únicas em temp: 3216


A comparação do número de chaves únicas (`key`) entre os diferentes datasets indica que nem todas as corridas de produção estão representadas de forma consistente em todas as fontes de dados. Essa diferença sugere a existência de corridas com dados ausentes em algum dos datasets, o que reforça a necessidade de esclarecer como esses casos devem ser tratados nas próximas etapas do projeto (ver Pergunta 3.1).

In [3]:
print("=== Dataset GAS ===")
print(gas.head())
print(gas.info())

print("\n=== Dataset TEMP ===")
print(temp.head())
print(temp.info())


=== Dataset GAS ===
,   key      Gas 1
,0    1  29.749986
,1    2  12.555561
,2    3  28.554793
,3    4  18.841219
,4    5   5.413692
,<class 'pandas.core.frame.DataFrame'>
,RangeIndex: 3239 entries, 0 to 3238
,Data columns (total 2 columns):
, #   Column  Non-Null Count  Dtype  
,---  ------  --------------  -----  
, 0   key     3239 non-null   int64  
, 1   Gas 1   3239 non-null   float64
,dtypes: float64(1), int64(1)
,memory usage: 50.7 KB
,None
,
,=== Dataset TEMP ===
,   key        Sampling time  Temperature
,0    1  2019-05-03 11:16:18       1571.0
,1    1  2019-05-03 11:25:53       1604.0
,2    1  2019-05-03 11:29:11       1618.0
,3    1  2019-05-03 11:30:01       1601.0
,4    1  2019-05-03 11:30:39       1613.0
,<class 'pandas.core.frame.DataFrame'>
,RangeIndex: 15907 entries, 0 to 15906
,Data columns (total 3 columns):
, #   Column         Non-Null Count  Dtype  
,---  ------         --------------  -----  
, 0   key            15907 non-null  int64  
, 1   Sampling time  159

In [4]:
# Verificar distribuição de ausentes por corrida
missing_per_key = temp.groupby('key')['Temperature'].apply(lambda x: x.isna().sum())
total_per_key = temp.groupby('key').size()

summary = pd.concat([missing_per_key, total_per_key], axis=1)
summary.columns = ['missing_count', 'total_count']

print("\nCorridas com dados ausentes:")
print("Corridas com pelo menos 1 valor ausente:", (summary['missing_count'] > 0).sum())
print("Corridas totalmente sem temperatura:", (summary['missing_count'] == summary['total_count']).sum())




,Corridas com dados ausentes:
,Corridas com pelo menos 1 valor ausente: 739
,Corridas totalmente sem temperatura: 0


## Conclusão inicial
A análise dos valores ausentes indica que 739 corridas de produção apresentam pelo menos uma medição de temperatura ausente. No entanto, nenhuma corrida está completamente sem registros de temperatura, o que permite a definição de uma varável alvo válida para todas as corridas.
Esse comportamento sugere que as ausências ocorrem de forma pontual ao longo do processo, reforçando a necessidade de definir criteriosamente qual medição de temperaturaserá utilizada como alvo do modelo(por exemplo a última medição disponível), bem como o tratamento adequado das medições intermediárias.

## Observações iniciais - Gás e Temperatura

1. **Consumo de gás**
O dataset de consumo de gás apresenta uma única observação por corrida de produção (`key`), indicando que o consumo de gás já se encontra agregado em nível de corrida. Não foram identificados valores ausentes nesse conjunto de dados, o que facilita sua integração com os demais datasets do projeto.

2. **Temperatura**
O dataset de temperatura contém múltiplas medições ao longo do tempo para cada corrida de produção (`key`). A coluna de tempo de amostragem está registrada como string e deverá ser convertida para o tipo `datetime` para permitir análises temporais.

Observa-se a presença de valores ausentes na coluna de temperatura, reforçando a necessidade de definir claramente qual medição será utilizada como variável alvo do modelo (por exemplo, a temperatura final do processo), bem como o tratamento adequado das medições intermediárias, a fim de evitar vazamento de informação.

### Observações iniciais - Aquecimento por arco elétrico

O dataset de aquecimento por arco elétrico contém registros em nível de evento, com múltiplas observações para a mesma corrida de produção ('key'). Cada registro representa um ciclo de aquecimento, com informações de tempo e potência elétrica. Para uso em modelagem, esses dados deverão ser agrupados por corrida, de forma a representar o consumo total de energia ao longo do processo. 
As colunas de início e fim do aquecimento estão registradas como strings e deverão ser convertidas para o tipo de 'datetime', permitindo o cálculo da duração de cada ciclo. 
As variáveis de potência ativa  e reativa representam o consumo energético durante os ciclos de aquecimento e poderão ser agregadas (por exemplo, soma, média ou agregações ponderadas pela duração) para representar o consumo total por corrida.

## Perguntas de esclarecimento

1 - Qual é exatamente a definição da temperatura que deve ser prevista (temperatura final do processo ou uma medição específica em determinado momento)?

2 - Existe uma coluna-chave que identifique cada corrida ou lote de produção em todos os datasets?

3 - Todas as variáveis disponíveis podem ser utilizadas como features, ou existem medições que ocorrem após a temperatura alvo (risco de data leakage)?

3.1 - Como devem ser tratadas as corridas de produção que possuem dados ausentes em algum dos datasets (por exemplo, presentes em arc, mas ausentes em gas ou temp)?

4 - Há requisistos específicos de precisão ou erro aceitável para o modelo do ponto de vista do negócio?

5 - Os dados possuem uma ordem temporal que deve ser respeitada na divisão entre treino e teste?

**Resposta da Revisora**

1 - Qual é exatamente a definição da temperatura que deve ser prevista (temperatura final do processo ou uma medição específica em determinado momento)?
**R: A temperatura alvo a ser prevista é a temperatura final da corrida de produção, medida ao final do processo.**
    
2 - Existe uma coluna-chave que identifique cada corrida ou lote de produção em todos os datasets?

**R: Sim. Existe uma coluna identificadora da corrida (run_id / heat_id) que está presente de forma consistente nos datasets (arc, gas e temp), permitindo o join entre eles por corrida de produção.**

3 - Todas as variáveis disponíveis podem ser utilizadas como features, ou existem medições que ocorrem após a temperatura alvo (risco de data leakage)?

**R:Nem todas as variáveis devem ser utilizadas como features. Apenas medições que ocorrem antes ou durante o processo, até o momento imediatamente anterior à medição da temperatura final, podem ser usadas. Medições realizadas após a temperatura alvo (ou que a utilizem diretamente no cálculo) devem ser excluídas para evitar data leakage.**

3.1 - Como devem ser tratadas as corridas de produção que possuem dados ausentes em algum dos datasets (por exemplo, presentes em arc, mas ausentes em gas ou temp)?

**R: No contexto atual, as corridas que aparecem em arc, mas estão ausentes em gas ou temp (ou vice-versa) devem ser removidas do conjunto final de modelagem, pois a ausência indica dados incompletos do processo e a imputação poderia introduzir viés físico e operacional** 


4 - Há requisistos específicos de precisão ou erro aceitável para o modelo do ponto de vista do negócio?

**R: Não há, neste momento, um limite de erro explicitamente definido pelo negócio.
O critério principal é minimizar o erro de previsão (ex.: RMSE) garantindo que o modelo seja estável, interpretável e útil para apoiar decisões operacionais.** 

5 - Os dados possuem uma ordem temporal que deve ser respeitada na divisão entre treino e teste?

**R: Não. Apesar de cada corrida ocorrer em um instante no tempo, as corridas são tratadas como observações independentes, sem dependência temporal direta entre elas. Portanto: uma divisão aleatória entre treino e teste é aceitável**

## Plano geral de resolução da tarefa (3-5 passos)

1. **Compreensão e preparação dos dados**

Analisar a estrutura da cada dataset, verificar valores ausentes, inconsistências e identificar a chave comum necessária para integrar os dados por corrida de produção.

3. **Análise exploratória dos dados (EDA)**

Investigar a distribuição das variáveis, relações entre consumo de energia, insumos e temperatura, além de identificar possíveis outliers ou padrões relevantes para o processo industrial.

5. **Construção do conjunto final de dados**

Agregar os dados em nível de corrida, combinando informações de consumo de energia elétrica, gás, duração do processo e insumos utilizados. Serão criadas features como consumo total, médias e agregações ponderadas pelo tempo, garantindo uma única linha por corrida.

6. **Modelagem preditiva**

Desenvolver modelos de regressão para prever a temperatura do metal, iniciando com um modelo baseline simples e avançando para modelos mais robustos, como regressões regularizadas e modelos baseados em árvores, avaliando o desempenho com métricas apropriadas.

9. **Avaliações e conclusões**

Avaliar o modelo final no conjunto de teste, interpretar os resultados e discutir como as previsões podem apoiar decisões para redução do consumo de energia e otimização do processo.

## Nota final

Este Notebook apresentou uma análise exploratória inicial dos dados disponíveis, bem como as principais questões e decisões a serem esclarecidas antes da modelagem. As observações aqui realizadas servem como base para a definição do escopo técnico e metodológico das próximas etapas do projeto.